# Patient-Specific Seizure Detection — LORO Evaluation

Fixed-hyperparameter pipeline (no SVM tuning). For each patient:
LORO over seizure records, per-fold operating-point calibration on
out-of-fold training scores, SzCORE event-level scoring.

**Set your hyperparameters once in the CONFIG cell.** AP / ROC-AUC are
reported per fold as leakage-free diagnostics only — they select nothing.

## Imports

In [37]:
# pip install timescoring

In [38]:
import os
# single-thread BLAS so the outer joblib parallelism over folds is clean
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"

from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import hashlib
from joblib import Parallel, delayed

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.svm import SVC
from sklearn.model_selection import (LeaveOneGroupOut, GroupKFold,
                                     StratifiedGroupKFold,  # CHANGED: stratified inner CV
                                     cross_val_predict)
from sklearn.metrics import roc_auc_score, average_precision_score

from timescoring import scoring
from timescoring.annotations import Annotation

## CONFIG

Everything you change between runs lives here. The SVM hyperparameters are
**fixed** — set them once; nothing is tuned on test data.

In [ ]:
# --- patients to run ---
# Sharding: set PATIENT_SHARD env var to run a subset across separate sessions.
#   PATIENT_SHARD="chb04"            -> just the heavy patient, its own session
#   PATIENT_SHARD="chb01,chb02,..."  -> a comma-separated list
#   unset / empty                    -> all 24
import os as _os
_ALL = [f"chb{n:02d}" for n in range(1, 25)]   # chb01 .. chb24
_shard = _os.environ.get("PATIENT_SHARD", "").strip()
if _shard:
    PATIENTS = [p.strip() for p in _shard.split(",") if p.strip()]
else:
    PATIENTS = _ALL

# --- paths ---
SEIZURE_DIR     = Path("Features/Seizure")
NON_SEIZURE_DIR = Path("Features/Non_Seizure")
ANNOTATIONS_CSV = "seizure_annotations.csv"
FEATURE_KEY     = "all_feats"      # dataset name inside the .mat files
LABEL_KEY       = "all_labels"
N_FEATURES      = 1530             # used only to detect transposed arrays

# --- model config ---
# C, gamma AND the feature-selection percentile are ALL auto-tuned per LORO
# fold (see C_GRID / GAMMA_GRID / PERC_GRID below). Nothing is set by hand;
# the empty cfg just carries the fold-tuned values.
cfg = {}

# --- windowing (must match feature extraction) ---
WINDOW_LEN = 10.0
STEP       = 2.5
GRID_FS    = 1

# --- operating-point calibration (threshold + persistence k) ---
THRESH_GRID       = None                       # None -> per-fold score quantiles
K_GRID            = [1, 2, 3, 4, 5, 6]         # consecutive-window persistence
FP_PER_DAY_BUDGET = 5                           # a-priori FP/day ceiling

# Inner CV used ONLY to pick the operating point (thr + k) per outer fold,
# computed on records that EXCLUDE the held-out test record -> leakage-free.
# GroupKFold keeps every record contributing to calibration (~5 fits/fold);
# set to None for full inner leave-one-record-out (clean but ~N fits/fold).
INNER_CAL_SPLITS  = 3

# === CHANGED (auto-tuned C/gamma per fold) ================================
# C, gamma and the feature-selection percentile are NO LONGER fixed. They are tuned PER LORO FOLD by
# an inner record-grouped CV (pooled average-precision) over that fold's own
# training pool, which already excludes the held-out record -> leakage-free.
# Nothing is set by hand: C, gamma and the feature-selection percentile are
# all picked per fold by the inner CV (leakage-free, same as C/gamma).
C_GRID        = [0.1, 1, 10, 100]
GAMMA_FACTORS = [1/3, 1, 3]
PERC_GRID     = [5,10, 25, 50]      # SelectPercentile %, tuned per fold
# =========================================================================

# --- SzCORE event-scoring parameters (Dan et al., 2024) ---
SZCORE_PARAMS = scoring.EventScoring.Parameters(
    toleranceStart=30, toleranceEnd=60, minOverlap=0,
    maxEventDuration=5 * 60, minDurationBetweenEvents=90,
)

# --- non-seizure TRAINING subsampling (speed; TRAINING ONLY, never test) ---
# Cap the interictal (label-0) windows used for TRAINING. Each non-seizure
# record contributes a CONTIGUOUS block (proportional to its length) so its
# windows stay in real-time order -- event-level calibration on these records
# (windows_to_mask / duration_of) then stays valid. ALL seizure windows are
# kept in full. Test records are scored in full elsewhere, so FP/day
# denominators are unaffected. None -> use all interictal windows.
# Deterministic per-record seed, stable across processes/shards.
# === CHANGED (combined 20k cap + seizure chunking) ========================
# OLD behavior: ALL seizure windows kept in full; only the non-seizure
# (label-0) windows were capped, at MAX_NONSEIZ_TRAIN_WINDOWS (10000).
# NEW behavior: BOTH classes are thinned into ONE combined budget.
#   - Each seizure record contributes a single CONTIGUOUS chunk that spans all
#     its seizures (first label-1 window to last) plus SEIZ_MARGIN_SEC of
#     interictal padding on EACH side (kept in real-time order).
#   - Non-seizure records then fill whatever remains up to MAX_TRAIN_WINDOWS.
# Test records are still scored in FULL (LORO) -- only TRAINING is thinned.
MAX_TRAIN_WINDOWS = 20000     # TOTAL training windows (seizure chunks + non-seiz)
SEIZ_MARGIN_SEC   = 300.0     # interictal padding kept each side of seizure span
# Kept defined for reference but no longer the driver (superseded above):
MAX_NONSEIZ_TRAIN_WINDOWS = 10000
SUBSAMPLE_SEED            = 0          # base seed (combined per record)
# ==========================================================================

# --- parallelism over LORO folds ---
OUTER_N_JOBS = 4

# --- outputs ---
SUMMARY_CSV = "RESULTS_SUMMARY.csv"   # combined summary across patients

SEIZ = pd.read_csv(ANNOTATIONS_CSV)


def make_pipe(cfg):
    # Memory caching removed on purpose (it wrote a growing 'cache/' folder
    # to disk and gave no speedup). cache_size below is the in-RAM libsvm
    # kernel buffer, per worker — not disk.
    return Pipeline([
        ("scale",  StandardScaler()),
        ("select", SelectPercentile(f_classif, percentile=cfg["percentile"])),
        ("svm",    SVC(kernel="rbf", C=cfg["C"], gamma=cfg["gamma"],
                       class_weight=None, cache_size=1024)),  # CHANGED: RAM-safe across workers
    ])

## Load one patient

Returns `seizure_records` and `non_records` dicts: `{record_name: (X, y)}`.

In [41]:
IMAG_TOL = 1e-6   # |imag| above this = invalid GC window, dropped


def _read_feats(h, key):
    """Read a features array, handling MATLAB-complex compound dtype.
    Returns (X_real_oriented_rows=windows, bad_window_mask)."""
    X = h[key][:]
    if X.dtype.names and "imag" in X.dtype.names:
        real, imag = X["real"], X["imag"]
    else:
        real, imag = X, np.zeros_like(X)
    if real.shape[0] == N_FEATURES:          # orient rows = windows
        real, imag = real.T, imag.T
    bad = np.abs(imag).max(axis=1) > IMAG_TOL
    return np.ascontiguousarray(real, dtype=np.float64), bad


def load_patient(patient):
    records = {}

    for f in sorted(SEIZURE_DIR.glob(f"{patient}*_seizure.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
                yv = h[LABEL_KEY][:].squeeze()
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if yv.shape[0] == X.shape[0]:
            y = yv.astype(int)
        else:
            y = yv[: X.shape[0]].astype(int)
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X, y = X[~bad], y[~bad]
        records[f.stem.replace("_seizure", "")] = (X, y)

    for f in sorted(NON_SEIZURE_DIR.glob(f"{patient}*_features.mat")):
        try:
            with h5py.File(f, "r") as h:
                X, bad = _read_feats(h, FEATURE_KEY)
        except OSError as e:
            print(f"  [SKIP corrupt] {f.name} — {e}")
            continue
        if bad.any():
            print(f"  [DROP complex] {f.name}: {int(bad.sum())} window(s) "
                  f"{list(np.where(bad)[0])}")
            X = X[~bad]
        y = np.zeros(X.shape[0], dtype=int)
        records[f.stem.replace("_features", "")] = (X, y)

    seizure_records = {k: v for k, v in records.items() if v[1].sum() > 0}
    non_records     = {k: v for k, v in records.items() if v[1].sum() == 0}
    return seizure_records, non_records

## Event-scoring & calibration helpers

In [ ]:
def duration_of(y):
    return (len(y) - 1) * STEP + WINDOW_LEN


def windows_to_mask(scores, duration, threshold, k):
    pos = (scores >= threshold).astype(int)
    alarm = np.zeros_like(pos); run = 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run >= k:
            alarm[i - k + 1: i + 1] = 1
    n = int(round(duration * GRID_FS))
    mask = np.zeros(n, dtype=bool)
    for i, a in enumerate(alarm):
        if a:
            s = int(round(i * STEP * GRID_FS))
            e = int(round((i * STEP + WINDOW_LEN) * GRID_FS))
            mask[s:min(e, n)] = True
    return mask, n


def ref_mask_from_record(record, n, offset_sec=0.0):
    # === CHANGED (seizure-chunk alignment): offset_sec shifts the ABSOLUTE
    # annotation times into the LOCAL coordinates of a subsampled seizure chunk
    # whose window 0 sits offset_sec into the file. Default 0.0 -> identical to
    # the old behavior for full records (test scoring, non-seizure records, and
    # chunks that were clipped to the file start).
    mask = np.zeros(n, dtype=bool)
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        s = int(round((r["start_sec"] - offset_sec) * GRID_FS))
        e = int(round((r["end_sec"]   - offset_sec) * GRID_FS))
        if e <= 0 or s >= n:
            continue
        mask[max(s, 0):min(e, n)] = True
    return mask


_REF_MASK_CACHE = {}

def event_counts(record_scores, threshold, k):
    # ref_mask_from_record never depends on (threshold, k) -- only on the
    # record's ground-truth annotations and its sample-grid length n.
    # calibrate() calls this ~80-150x per patient/config with the SAME
    # record_scores (same n per record each time), so caching by
    # (record, n) avoids rebuilding an identical mask from the annotations
    # CSV every single combo. Pure speed fix -- ref mask is ground truth,
    # not a result, so caching it changes nothing about correctness.
    TP = FP = FN = 0
    # === CHANGED: each OOF record now carries a 5th item, offset_sec, used to
    # align the reference mask to a subsampled seizure chunk's local grid.
    for record, scores, y, dur, offset_sec in record_scores:
        hyp, n = windows_to_mask(scores, dur, threshold, k)
        key = (record, n, offset_sec)
        ref = _REF_MASK_CACHE.get(key)
        if ref is None:
            ref = ref_mask_from_record(record, n, offset_sec)
            _REF_MASK_CACHE[key] = ref
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
    return TP, FP, FN


def _arming_times(sc, thr, k):
    pos = (sc >= thr).astype(int)
    arm, run = [], 0
    for i, p in enumerate(pos):
        run = run + 1 if p else 0
        if run == k:
            arm.append(i * STEP + WINDOW_LEN)
    return arm


def detection_latencies(record, sc, thr, k):
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    lat = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        if cands:
            lat.append(min(cands) - onset)
    return lat


def _safe_div(a, b):
    return a / b if b else float("nan")


def segment_metrics(y, sc, thr):
    y = np.asarray(y).astype(int)
    pred = (sc >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum())
    fn = int(((pred == 0) & (y == 1)).sum())
    acc  = _safe_div(tp + tn, tp + tn + fp + fn)
    spec = _safe_div(tn, tn + fp)
    sens = _safe_div(tp, tp + fn)
    prec = _safe_div(tp, tp + fp)
    f1 = (_safe_div(2 * prec * sens, prec + sens)
          if (prec == prec and sens == sens) else float("nan"))
    return acc, spec, sens, prec, f1, tp, fp, fn


def calibrate(oof_record_scores):
    # Selects threshold + k (the OPERATING POINT) on out-of-fold train scores.
    # This is not model selection — the SVM is already fixed.
    # === CHANGED: OOF tuples are now 5-wide (.., offset_sec); unpack to match.
    total_days = sum(d for _, _, _, d, _ in oof_record_scores) / 86400
    all_scores = np.concatenate([s for _, s, _, _, _ in oof_record_scores])
    if THRESH_GRID is None:
        thr_grid = np.unique(np.quantile(all_scores, np.linspace(0.05, 0.999, 30)))
    else:
        thr_grid = THRESH_GRID

    best = best_feasible = None
    for thr in thr_grid:
        for k in K_GRID:
            TP, FP, FN = event_counts(oof_record_scores, thr, k)
            sens = TP / (TP + FN) if (TP + FN) else 0.0
            prec = TP / (TP + FP) if (TP + FP) else 0.0
            f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
            fp_day = FP / total_days if total_days else 0.0
            feas_key = (sens, -k, f1, -fp_day)
            if fp_day <= FP_PER_DAY_BUDGET:
                if best_feasible is None or feas_key > best_feasible[0]:
                    best_feasible = (feas_key, thr, k)
            fb_key = (f1, sens, -k)
            if best is None or fb_key > best[0]:
                best = (fb_key, thr, k)
    chosen = best_feasible if best_feasible is not None else best
    return chosen[1], chosen[2]


def event_detections(record, sc, thr, k):
    """One row per ground-truth seizure EVENT in `record`: whether the
    operating point (thr, k) caught it and, if so, the onset latency.
    Uses the SAME arming logic / tolerance window as detection_latencies,
    so the flag and latency stay consistent with the reported metrics."""
    TOL_PRE, TOL_POST = 30, 60
    arm_times = _arming_times(sc, thr, k)
    rows = []
    for _, r in SEIZ[SEIZ["file"] == record].iterrows():
        onset, offset = r["start_sec"], r["end_sec"]
        cands = [t for t in arm_times
                 if (onset - TOL_PRE) <= t <= (offset + TOL_POST)]
        rows.append({
            "event_start": onset,
            "event_end":   offset,
            "detected":    1 if cands else 0,
            "latency_s":   (min(cands) - onset) if cands else float("nan"),
        })
    return rows


## LORO for one patient

Fixed SVM. Per fold: hold out one record, compute the calibration OOF on the
remaining records (inner grouped CV that EXCLUDES the held-out record), pick
(threshold, k) from those leakage-free scores, fit the deployment model on all
remaining records, and score the held-out record. Seizure records drive
sensitivity (Loop 1); seizure-free records get the same true leave-one-record-out
treatment (Loop 2) so their false alarms count.

`ap` / `roc_auc` are computed per fold on the held-out record as **diagnostics
only** — they are printed, never used to choose anything.

In [ ]:
def _stable_seed(*parts):
    """Deterministic 64-bit seed from arbitrary parts, stable ACROSS processes
    (unlike Python's hash(), which is salted per-process by PYTHONHASHSEED)."""
    s = "|".join("" if p is None else str(p) for p in parts)
    return int.from_bytes(hashlib.blake2b(s.encode(), digest_size=8).digest(),
                          "little")


def build_train(seizure_records, non_records=None, exclude=None,
                exclude_non=None):
    """Stack TRAINING data (used for model fits and OOF calibration only —
    never for scoring a test record).

    seizure_records : dict of seizure-containing records {name:(X,y)}
    non_records     : dict of seizure-free records {name:(X,y)} or None
    exclude         : seizure record name to leave OUT of training
    exclude_non     : non-seizure record name to leave OUT of training
    Groups are the record names (used for OOF calibration CV).

    If MAX_NONSEIZ_TRAIN_WINDOWS is set, the interictal (label-0) windows placed
    into TRAINING are capped at that total. Each included non-seizure record
    contributes a CONTIGUOUS block sized proportionally to its length, with a
    deterministic per-record seed. Contiguous (not scattered) keeps the windows
    in real-time order, so windows_to_mask's i*STEP mapping and duration_of stay
    valid when these records feed the event-level calibration. ALL seizure
    windows are always kept. This only thins training; test records are scored
    in full elsewhere, so sensitivity, FP/day, and the LORO record boundaries
    are unaffected.
    """
    Xs, ys, grps = [], [], []
    offsets = {}   # === CHANGED: record_name -> seconds the kept block starts into the file

    # === CHANGED (seizure chunking) ======================================
    # OLD: every seizure window was kept in full. NEW: each seizure record is
    # reduced to ONE contiguous chunk spanning all its seizures (first label-1
    # window to last) plus SEIZ_MARGIN_SEC of interictal on each side. The
    # chunk's start offset (seconds) is recorded so the event-level calibration
    # can realign the absolute annotations (see ref_mask_from_record /
    # event_counts). The chunk stays contiguous -> windows_to_mask / duration_of
    # remain valid. Test records are scored in full elsewhere, unaffected.
    margin_w = int(round(SEIZ_MARGIN_SEC / STEP))
    total_seiz_kept = 0
    for name, (X, y) in seizure_records.items():
        if name == exclude:
            continue
        pos = np.where(y == 1)[0]
        if pos.size == 0:
            # no labeled seizure (shouldn't happen for a seizure record) -> keep full
            Xs.append(X); ys.append(y); grps.extend([name] * len(y))
            offsets[name] = 0.0
            total_seiz_kept += len(y)
            continue
        start = max(0, int(pos.min()) - margin_w)
        end   = min(len(y), int(pos.max()) + 1 + margin_w)
        Xc, yc = X[start:end], y[start:end]
        Xs.append(Xc); ys.append(yc); grps.extend([name] * len(yc))
        offsets[name] = start * STEP            # window 0 of chunk = start*STEP sec
        total_seiz_kept += len(yc)
    # =====================================================================

    if non_records:
        included = [(name, X, y) for name, (X, y) in non_records.items()
                    if name != exclude_non]
        total_non = sum(len(y) for _, _, y in included)
        # === CHANGED: non-seizure budget = remainder of the combined cap after
        # the seizure chunks (was a fixed MAX_NONSEIZ_TRAIN_WINDOWS).
        cap = (None if MAX_TRAIN_WINDOWS is None
               else max(0, MAX_TRAIN_WINDOWS - total_seiz_kept))

        if cap is None or total_non <= cap:
            # no thinning needed
            for name, X, y in included:
                Xs.append(X); ys.append(y); grps.extend([name] * len(y))
                offsets[name] = 0.0
        elif cap == 0:
            # seizure chunks already fill the budget -> add no interictal
            pass
        else:
            # proportional per-record cap as a CONTIGUOUS block, seeded
            # deterministically per (fold, record) and stable across processes.
            for name, X, y in included:
                n_i = len(y)
                keep_i = max(1, int(round(cap * n_i / total_non)))
                if keep_i >= n_i:
                    idx = np.arange(n_i)
                else:
                    rng = np.random.default_rng(
                        _stable_seed(SUBSAMPLE_SEED, exclude, exclude_non, name))
                    start = int(rng.integers(0, n_i - keep_i + 1))
                    idx = np.arange(start, start + keep_i)   # contiguous block
                Xs.append(X[idx]); ys.append(y[idx])
                grps.extend([name] * len(idx))
                offsets[name] = 0.0   # non-seizure file has no annotation to align

    # === CHANGED: also return per-record offsets for calibration alignment
    return np.vstack(Xs), np.concatenate(ys), np.array(grps), offsets


def _inner_cv(grps):
    """=== ADDED: record-grouped, class-STRATIFIED inner CV, shared by the
    C/gamma tuning AND the thr/k OOF calibration. StratifiedGroupKFold keeps
    each whole record on one side of every split (no record's windows span
    train+val) while balancing the seizure (label-1) proportion across splits.
    Spreading the few seizure records across folds (instead of letting them
    clump, as plain GroupKFold could) also lowers the chance of an all-interictal
    training split. n_splits is capped at the number of records; deterministic
    (no shuffle). Falls back to leave-one-record-out only in the degenerate
    <2-record case."""
    n_groups = len(np.unique(grps))
    if n_groups < 2:
        return LeaveOneGroupOut()
    k = (n_groups if (INNER_CAL_SPLITS is None or n_groups <= INNER_CAL_SPLITS)
         else INNER_CAL_SPLITS)
    return StratifiedGroupKFold(n_splits=k)


def oof_train_record_scores(seizure_records, non_records, cfg,
                            exclude=None, exclude_non=None):
    """Out-of-fold decision scores per record over the training pool that
    EXCLUDES the held-out record. Inner CV is grouped by record (GroupKFold,
    or leave-one-record-out when groups are few), so no record scores itself
    AND the held-out record never enters any inner training fold -> the
    resulting (thr, k) is leakage-free. Used only to calibrate the operating
    point.
    """
    # === CHANGED: build_train now also returns per-record start offsets
    Xtr, ytr, grps, offsets = build_train(seizure_records, non_records,
                                 exclude=exclude, exclude_non=exclude_non)
    cv = _inner_cv(grps)   # CHANGED: StratifiedGroupKFold (record-grouped + stratified)
    oof = cross_val_predict(make_pipe(cfg), Xtr, ytr, groups=grps,
                            cv=cv, method="decision_function", n_jobs=1)
    out = []
    for name in np.unique(grps):
        m = (grps == name)
        # === CHANGED: append offset_sec so calibration realigns chunked seizures
        out.append((name, oof[m], ytr[m], duration_of(ytr[m]),
                    offsets.get(name, 0.0)))
    return out


def diag_scores(y, sc):
    y = np.asarray(y).astype(int)
    if y.sum() == 0 or y.sum() == len(y):
        return float("nan"), float("nan")
    return roc_auc_score(y, sc), average_precision_score(y, sc)



def tune_cfg(Xtr, ytr, grps, cfg):
    """=== ADDED (per-fold C/gamma tuning) ==============================
    Choose (C, gamma) for THIS fold by an inner record-grouped CV over the
    fold's training pool (Xtr already EXCLUDES the outer held-out record ->
    leakage-free). For each grid point we take out-of-fold decision scores via
    cross_val_predict grouped by record (no record scores itself), then rank
    grid points by a SINGLE POOLED average-precision over all OOF scores.
    Pooling (rather than averaging a per-fold AP) keeps the metric defined even
    when an inner fold happens to contain no seizure record. C, gamma and the
    feature-selection percentile are all searched. Returns a cfg COPY with the winner.
    """
    if len(np.unique(ytr)) < 2:
        return {**cfg, "C": 1, "gamma": "scale", "percentile": 15}, np.zeros(len(ytr))
    # tuning-only subsample (ALL seizure windows kept; interictal thinned)
    Xt, yt, gt = _subsample_for_tuning(
        Xtr, ytr, grps, TUNE_MAX_WINDOWS,
        seed=_stable_seed(SUBSAMPLE_SEED, "tune", len(ytr)))
    cv = _inner_cv(gt)   # StratifiedGroupKFold (record-grouped + stratified)
    best = None   # (ap, C, gamma_value, percentile, oof_scores)
    n_in = Xt.shape[1]                        # feature count before ANOVA selection
    for C in C_GRID:
        for fac in GAMMA_FACTORS:             # gamma = fac * 'scale' (~ fac / d)
            for p in PERC_GRID:
                d = max(1, int(round(p / 100 * n_in)))   # ~features kept by SelectPercentile
                gval = fac / d                            # 'scale' ~ 1/d after StandardScaler
                pipe = make_pipe({**cfg, "C": C, "gamma": gval, "percentile": p})
                oof = cross_val_predict(pipe, Xt, yt, groups=gt, cv=cv,
                                        method="decision_function", n_jobs=1)
                ap = average_precision_score(yt, oof)
                if best is None or ap > best[0]:
                    best = (ap, C, gval, p, oof)
    # === CHANGED (speed): also return the winning combo's OOF scores so the
    # thr/k calibration can REUSE them instead of recomputing an identical
    # cross_val_predict (same data + same deterministic CV). Result-identical;
    # saves ~3 SVM fits and one build_train per fold.
    return {**cfg, "C": best[1], "gamma": best[2], "percentile": best[3]}, best[4]


def _oof_by_record(oof_scores, ytr, grps, offsets):
    """=== ADDED: assemble per-record OOF tuples (name, scores, y, duration,
    offset_sec) from a precomputed OOF vector -- the shape calibrate() expects.
    Lets process_*_fold REUSE tune_cfg's OOF instead of recomputing it."""
    out = []
    for name in np.unique(grps):
        m = (grps == name)
        out.append((name, oof_scores[m], ytr[m], duration_of(ytr[m]),
                    offsets.get(name, 0.0)))
    return out


def process_seizure_fold(seizure_records, non_records, test_record, cfg):
    """LOOP 1 fold: train on all non-seizure + all-but-one seizure, test on the
    held-out seizure record. C/gamma are tuned on this fold's training pool,
    then the operating point (thr, k) is calibrated on a nested OOF computed
    WITHOUT this record -> both are leakage-free per fold."""
    # === CHANGED (speed): build the pool ONCE; tune C/gamma and REUSE its OOF
    # for the thr/k calibration (no second build_train / cross_val_predict).
    Xtr, ytr, grps, offsets = build_train(seizure_records, non_records,
                                          exclude=test_record)
    cfg, oof_scores = tune_cfg(Xtr, ytr, grps, cfg)
    thr, k = calibrate(_oof_by_record(oof_scores, ytr, grps, offsets))
    model = make_pipe(cfg).fit(Xtr, ytr)
    Xte, yte = seizure_records[test_record]
    sc = model.decision_function(Xte)
    roc, ap = diag_scores(yte, sc)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, roc=roc, ap=ap,
                C=cfg["C"], gamma=cfg["gamma"], percentile=cfg["percentile"])  # CHANGED: tuned


def process_nonseizure_fold(seizure_records, non_records, test_record, cfg):
    """LOOP 2 fold (TRUE LONO): train on all seizure + all-but-one non-seizure,
    test on the held-out non-seizure record. C/gamma tuned on this fold's pool;
    model refit per record (no FP leak); (thr, k) calibrated on a nested OOF
    computed WITHOUT this record."""
    # === CHANGED (speed): build the pool ONCE; tune C/gamma and REUSE its OOF
    # for the thr/k calibration (no second build_train / cross_val_predict).
    Xtr, ytr, grps, offsets = build_train(seizure_records, non_records,
                                          exclude_non=test_record)
    cfg, oof_scores = tune_cfg(Xtr, ytr, grps, cfg)
    thr, k = calibrate(_oof_by_record(oof_scores, ytr, grps, offsets))
    model = make_pipe(cfg).fit(Xtr, ytr)
    Xte, yte = non_records[test_record]
    sc = model.decision_function(Xte)
    return dict(record=test_record, sc=sc, y=yte, dur=duration_of(yte),
                thr=thr, k=k, roc=float("nan"), ap=float("nan"),
                C=cfg["C"], gamma=cfg["gamma"], percentile=cfg["percentile"])  # CHANGED: tuned


## Run one patient (build tables + return summary row)

In [44]:
def run_patient(patient, cfg):
    seizure_records, non_records = load_patient(patient)
    if not seizure_records:
        print(f"[{patient}] no seizure records — skipping")
        return None, None, None, None
    print(f"[{patient}] seizure={len(seizure_records)} "
          f"non-seizure={len(non_records)}")

    # --- LOOP 1: LORO over seizure records (sensitivity / latency) ---
    # Each fold computes its OWN calibration OOF on records that EXCLUDE the
    # held-out test record (nested) -> the operating point is leakage-free.
    seiz_results = Parallel(n_jobs=OUTER_N_JOBS)(
        delayed(process_seizure_fold)(seizure_records, non_records, r, cfg)
        for r in seizure_records.keys()
    )

    test_outputs, meta = [], {}
    for res in seiz_results:
        test_outputs.append((res["record"], res["sc"], res["y"], res["dur"],
                             res["thr"], res["k"]))
        # === CHANGED: store per-fold tuned C/gamma
        meta[res["record"]] = dict(type="seizure", roc=res["roc"], ap=res["ap"],
                                   C=res["C"], gamma=res["gamma"], percentile=res["percentile"])
        print(f"  [SEIZ] {res['record']}  thr={res['thr']:.3f} k={res['k']}  "
              f"C={res['C']} g={res['gamma']} p={res['percentile']}  "          # CHANGED: tuned hp
              f"roc={res['roc']:.3f} ap={res['ap']:.3f}  "
              f"test_pos={int(res['y'].sum())}")

    # --- LOOP 2: TRUE LONO over non-seizure records (specificity / FP) ---
    if non_records:
        non_results = Parallel(n_jobs=OUTER_N_JOBS)(
            delayed(process_nonseizure_fold)(seizure_records, non_records, r, cfg)
            for r in non_records.keys()
        )
        for res in non_results:
            test_outputs.append((res["record"], res["sc"], res["y"],
                                 res["dur"], res["thr"], res["k"]))
            # === CHANGED: store per-fold tuned C/gamma
            meta[res["record"]] = dict(type="nonseizure", roc=float("nan"),
                                       ap=float("nan"),
                                       C=res["C"], gamma=res["gamma"], percentile=res["percentile"])
        print(f"  [NON] {len(non_results)} records scored (true LONO)")

    # --- per-record pass: segment table + event totals ---
    TP = FP = FN = 0
    all_latencies, segment_rows, event_rows, fp_rows = [], [], [], []
    for record, sc, y, dur, thr, k in test_outputs:
        is_seiz = meta[record]["type"] == "seizure"
        hyp, n = windows_to_mask(sc, dur, thr, k)
        ref = ref_mask_from_record(record, n)
        s = scoring.EventScoring(Annotation(ref, GRID_FS),
                                 Annotation(hyp, GRID_FS), SZCORE_PARAMS)
        TP += s.tp; FP += s.fp; FN += s.refTrue - s.tp
        # === false-positive table: one row per hypothesis (alarm) event that
        # SzCORE counts as a false detection. Mirrors the library's own FP
        # loop (hyp events not overlapping tpMask) so counts match s.fp.
        for _ev in s.hyp.events:
            _a, _b = round(_ev[0] * s.fs), round(_ev[1] * s.fs)
            if np.all(~s.tpMask[_a:_b]):
                fp_rows.append({
                    "patient": patient, "record": record,
                    "fp_start": _ev[0], "fp_duration_s": _ev[1] - _ev[0],
                })
        if is_seiz:
            all_latencies.extend(detection_latencies(record, sc, thr, k))
            # === event-level table: one row per ground-truth seizure event
            for ev in event_detections(record, sc, thr, k):
                event_rows.append({
                    "patient": patient, "record": record,
                    "event_start": ev["event_start"],
                    "event_end":   ev["event_end"],
                    "detected":    ev["detected"],
                    "latency_s":   ev["latency_s"],
                    "threshold": thr, "k": k,
                    "percentile": meta[record]["percentile"],
                })

        acc, spec, sens, prec, f1, stp, sfp, sfn = segment_metrics(y, sc, 0.0)
        if not is_seiz:
            acc = spec = sens = prec = f1 = np.nan
        segment_rows.append({
            # === CHANGED: 'config' column dropped (constant label, no info)
            "C": meta[record]["C"],
            "gamma": meta[record]["gamma"],
            "percentile": meta[record]["percentile"],
            "patient": patient, "record": record,
            "n_windows": int(len(y)), "n_seizure_windows": int(np.sum(y)),
            "record_length": dur, "threshold": thr, "k": k,
            "roc_auc": meta[record]["roc"], "ap": meta[record]["ap"],
            "seg_accuracy": acc, "seg_specificity": spec,
            "seg_sensitivity": sens, "seg_precision": prec, "seg_f1": f1,
            "tp": stp, "fp": sfp, "fn": sfn,
        })
        kind = "SEIZ" if is_seiz else "NON"
        print(f"    {record} done  ({kind})", flush=True)

    # --- event-level totals ---
    sens = TP/(TP+FN) if (TP+FN) else 0.0
    prec = TP/(TP+FP) if (TP+FP) else 0.0
    f1   = 2*sens*prec/(sens+prec) if (sens+prec) else 0.0
    total_len_sec = sum(t[3] for t in test_outputs)
    days = total_len_sec / 86400
    med_lat = float(np.median(all_latencies)) if all_latencies else float("nan")

    fp_day = FP / days if days else float("nan")
    print(f"  EVENT: TP={TP} FP={FP} FN={FN}  sens={sens:.3f} "
          f"prec={prec:.3f} f1={f1:.3f}  FP/day={fp_day:.3f}  "
          f"lat={med_lat:.1f}s")

    # === CHANGED: C/gamma now vary per fold -> report the MODAL choice for the
    # patient (per-fold values are in the segment CSV).
    _cs = [m.get("C") for m in meta.values() if m.get("C") is not None]
    _gs = [m.get("gamma") for m in meta.values() if m.get("gamma") is not None]
    mode_C     = max(set(_cs), key=_cs.count) if _cs else None
    mode_gamma = max(set(_gs), key=_gs.count) if _gs else None
    _ps = [m.get("percentile") for m in meta.values() if m.get("percentile") is not None]
    mode_perc  = max(set(_ps), key=_ps.count) if _ps else None

    summary_row = {
        # === CHANGED: 'config' column dropped (constant label, no info)
        "C": mode_C, "gamma": mode_gamma,
        "percentile": mode_perc,
        "patient": patient,
        "total_length_hours": total_len_sec / 3600,
        "TP": TP, "FP": FP, "FN": FN,
        "sensitivity": sens, "precision": prec, "f1": f1,
        "FP/day": FP / days if days else float("nan"),
        "FP/hour": FP / (days * 24) if days else float("nan"),
        "latency_median": med_lat, "n_detected": len(all_latencies),
        "mean_fold_roc_auc": float(np.nanmean(
            [m["roc"] for m in meta.values()])),
        "mean_fold_ap": float(np.nanmean([m["ap"] for m in meta.values()])),
    }
    return summary_row, segment_rows, event_rows, fp_rows


# Run

Auto-tuned C/gamma per fold. Runs every patient in `PATIENTS` (set the `PATIENT_SHARD` env var to subset).

## chb01

In [ ]:
PATIENT = "chb01"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb04
  [DROP complex] chb04_12_features.mat: 2 window(s) [np.int64(95), np.int64(135)]
[chb04] seizure=3 non-seizure=39
  [SEIZ] chb04_05  thr=-1.023 k=8  C=1 g=scale p=100  roc=0.943 ap=0.406  test_pos=20
  [SEIZ] chb04_08  thr=-0.780 k=2  C=1 g=scale p=100  roc=0.895 ap=0.129  test_pos=44
  [SEIZ] chb04_28  thr=-0.162 k=2  C=10 g=scale p=10  roc=0.759 ap=0.051  test_pos=88


In [ ]:
PATIENT = "chb02"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb05
[chb05] seizure=5 non-seizure=34
  [SEIZ] chb05_06  thr=-1.401 k=8  C=10 g=0.001 p=10  roc=0.977 ap=0.808  test_pos=46
  [SEIZ] chb05_13  thr=-1.147 k=8  C=10 g=0.001 p=10  roc=0.989 ap=0.882  test_pos=44
  [SEIZ] chb05_16  thr=-1.209 k=6  C=10 g=0.001 p=10  roc=0.962 ap=0.747  test_pos=39
  [SEIZ] chb05_17  thr=-1.175 k=6  C=10 g=0.001 p=10  roc=0.916 ap=0.459  test_pos=48
  [SEIZ] chb05_22  thr=-1.211 k=6  C=10 g=0.001 p=10  roc=0.981 ap=0.836  test_pos=47
  [NON] 34 records scored (true LONO)
    chb05_06 done  (SEIZ)
    chb05_13 done  (SEIZ)
    chb05_16 done  (SEIZ)
    chb05_17 done  (SEIZ)
    chb05_22 done  (SEIZ)
    chb05_01 done  (NON)
    chb05_02 done  (NON)
    chb05_03 done  (NON)
    chb05_04 done  (NON)
    chb05_05 done  (NON)
    chb05_07 done  (NON)
    chb05_08 done  (NON)
    chb05_09 done  (NON)
    chb05_10 done  (NON)
    chb05_11 done  (NON)
    chb05_12 done  (NON)
    chb05_14 done  (NON)
    ch

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb05,50.336806,5,6,0,1.0,0.454545,0.625,2.86073,0.119197,31.5,5,0.965061,0.746545


In [ ]:
PATIENT = "chb03"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb06
[chb06] seizure=7 non-seizure=11
  [SEIZ] chb06_01  thr=-1.001 k=8  C=10 g=0.0001 p=5  roc=0.869 ap=0.047  test_pos=19
  [SEIZ] chb06_04  thr=-1.002 k=6  C=1 g=0.0001 p=10  roc=0.775 ap=0.017  test_pos=16
  [SEIZ] chb06_09  thr=-0.232 k=2  C=10 g=0.01 p=10  roc=0.923 ap=0.015  test_pos=7
  [SEIZ] chb06_10  thr=-1.161 k=6  C=10 g=0.01 p=10  roc=0.871 ap=0.283  test_pos=5
  [SEIZ] chb06_13  thr=-0.135 k=2  C=10 g=0.01 p=10  roc=0.988 ap=0.432  test_pos=5
  [SEIZ] chb06_18  thr=-0.107 k=2  C=10 g=0.01 p=10  roc=0.911 ap=0.015  test_pos=5
  [SEIZ] chb06_24  thr=-0.141 k=2  C=10 g=0.01 p=10  roc=0.854 ap=0.008  test_pos=7
  [NON] 11 records scored (true LONO)
    chb06_01 done  (SEIZ)
    chb06_04 done  (SEIZ)
    chb06_09 done  (SEIZ)
    chb06_10 done  (SEIZ)
    chb06_13 done  (SEIZ)
    chb06_18 done  (SEIZ)
    chb06_24 done  (SEIZ)
    chb06_02 done  (NON)
    chb06_03 done  (NON)
    chb06_05 done  (NON)
    chb06_06 done

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.01,10,chb06,80.347917,3,16,7,0.3,0.157895,0.206897,4.779215,0.199134,58.5,3,0.884376,0.116667


In [ ]:
PATIENT = "chb04"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb07
[chb07] seizure=3 non-seizure=16
  [SEIZ] chb07_12  thr=-0.344 k=2  C=10 g=0.001 p=5  roc=0.769 ap=0.058  test_pos=35
  [SEIZ] chb07_13  thr=-1.036 k=6  C=10 g=0.01 p=10  roc=0.819 ap=0.525  test_pos=39
  [SEIZ] chb07_19  thr=-0.945 k=8  C=1 g=0.01 p=10  roc=0.835 ap=0.244  test_pos=76
  [NON] 16 records scored (true LONO)
    chb07_12 done  (SEIZ)
    chb07_13 done  (SEIZ)
    chb07_19 done  (SEIZ)
    chb07_01 done  (NON)
    chb07_02 done  (NON)
    chb07_03 done  (NON)
    chb07_04 done  (NON)
    chb07_05 done  (NON)
    chb07_06 done  (NON)
    chb07_07 done  (NON)
    chb07_08 done  (NON)
    chb07_09 done  (NON)
    chb07_10 done  (NON)
    chb07_11 done  (NON)
    chb07_14 done  (NON)
    chb07_15 done  (NON)
    chb07_16 done  (NON)
    chb07_17 done  (NON)
    chb07_18 done  (NON)
  EVENT: TP=2 FP=11 FN=1  sens=0.667 prec=0.154 f1=0.250  FP/day=3.049  lat=26.2s
  >>> SAVED [chb07] -> RESULTS_SUMMARY_chb07.csv

Wr

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.01,10,chb07,86.599306,2,11,1,0.666667,0.153846,0.25,3.048523,0.127022,26.25,2,0.807539,0.275589


In [ ]:
PATIENT = "chb05"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb08
[chb08] seizure=5 non-seizure=15
  [SEIZ] chb08_02  thr=0.654 k=1  C=10 g=0.001 p=10  roc=0.828 ap=0.473  test_pos=69
  [SEIZ] chb08_05  thr=-0.932 k=8  C=10 g=0.001 p=10  roc=0.931 ap=0.743  test_pos=76
  [SEIZ] chb08_11  thr=0.610 k=2  C=10 g=0.001 p=10  roc=0.957 ap=0.789  test_pos=53
  [SEIZ] chb08_13  thr=-0.820 k=1  C=0.1 g=0.01 p=5  roc=0.951 ap=0.648  test_pos=64
  [SEIZ] chb08_21  thr=-0.966 k=8  C=10 g=0.001 p=10  roc=0.766 ap=0.507  test_pos=105
  [NON] 15 records scored (true LONO)
    chb08_02 done  (SEIZ)
    chb08_05 done  (SEIZ)
    chb08_11 done  (SEIZ)
    chb08_13 done  (SEIZ)
    chb08_21 done  (SEIZ)
    chb08_03 done  (NON)
    chb08_04 done  (NON)
    chb08_10 done  (NON)
    chb08_12 done  (NON)
    chb08_14 done  (NON)
    chb08_15 done  (NON)
    chb08_16 done  (NON)
    chb08_17 done  (NON)
    chb08_18 done  (NON)
    chb08_19 done  (NON)
    chb08_20 done  (NON)
    chb08_22 done  (NON)
    chb0

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb08,21.008333,4,3,1,0.8,0.571429,0.666667,3.427211,0.1428,22.25,4,0.886563,0.631813


## chb06

In [ ]:
PATIENT = "chb06"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb09
[chb09] seizure=3 non-seizure=16
  [SEIZ] chb09_06  thr=-0.121 k=2  C=10 g=0.001 p=10  roc=0.963 ap=0.737  test_pos=26
  [SEIZ] chb09_08  thr=-0.998 k=2  C=0.1 g=0.0001 p=10  roc=0.957 ap=0.582  test_pos=60
  [SEIZ] chb09_19  thr=0.296 k=1  C=10 g=0.01 p=5  roc=0.953 ap=0.393  test_pos=25
  [NON] 16 records scored (true LONO)
    chb09_06 done  (SEIZ)
    chb09_08 done  (SEIZ)
    chb09_19 done  (SEIZ)
    chb09_01 done  (NON)
    chb09_02 done  (NON)
    chb09_03 done  (NON)
    chb09_04 done  (NON)
    chb09_05 done  (NON)
    chb09_07 done  (NON)
    chb09_09 done  (NON)
    chb09_10 done  (NON)
    chb09_11 done  (NON)
    chb09_12 done  (NON)
    chb09_13 done  (NON)
    chb09_14 done  (NON)
    chb09_15 done  (NON)
    chb09_16 done  (NON)
    chb09_17 done  (NON)
    chb09_18 done  (NON)
  EVENT: TP=4 FP=7 FN=0  sens=1.000 prec=0.364 f1=0.533  FP/day=2.015  lat=26.5s
  >>> SAVED [chb09] -> RESULTS_SUMMARY_chb09.csv



,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb09,83.364583,4,7,0,1.0,0.363636,0.533333,2.015244,0.083969,26.5,4,0.957723,0.570809


In [ ]:
PATIENT = "chb07"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb10
[chb10] seizure=7 non-seizure=18
  [SEIZ] chb10_12  thr=-1.045 k=6  C=10 g=0.001 p=10  roc=0.991 ap=0.336  test_pos=14
  [SEIZ] chb10_20  thr=-1.239 k=8  C=10 g=0.001 p=10  roc=0.979 ap=0.817  test_pos=28
  [SEIZ] chb10_27  thr=-1.008 k=8  C=10 g=0.001 p=10  roc=0.975 ap=0.625  test_pos=26
  [SEIZ] chb10_30  thr=-1.089 k=6  C=10 g=0.001 p=10  roc=0.909 ap=0.062  test_pos=23
  [SEIZ] chb10_31  thr=-1.118 k=6  C=10 g=0.001 p=10  roc=0.998 ap=0.920  test_pos=30
  [SEIZ] chb10_38  thr=-1.047 k=8  C=10 g=0.001 p=10  roc=0.955 ap=0.417  test_pos=35
  [SEIZ] chb10_89  thr=1.720 k=1  C=10 g=0.001 p=10  roc=0.952 ap=0.487  test_pos=21
  [NON] 18 records scored (true LONO)
    chb10_12 done  (SEIZ)
    chb10_20 done  (SEIZ)
    chb10_27 done  (SEIZ)
    chb10_30 done  (SEIZ)
    chb10_31 done  (SEIZ)
    chb10_38 done  (SEIZ)
    chb10_89 done  (SEIZ)
    chb10_01 done  (NON)
    chb10_02 done  (NON)
    chb10_03 done  (NON)
    chb1

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb10,56.690278,5,15,2,0.714286,0.25,0.37037,6.350295,0.264596,19.5,5,0.965626,0.523467


In [ ]:
PATIENT = "chb08"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb09"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb10"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


## chb11

In [ ]:
PATIENT = "chb11"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb12"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb13"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb14"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb15"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb18
[chb18] seizure=6 non-seizure=30
  [SEIZ] chb18_29  thr=-1.274 k=8  C=10 g=0.0001 p=5  roc=0.890 ap=0.669  test_pos=20
  [SEIZ] chb18_30  thr=-1.192 k=8  C=10 g=0.001 p=10  roc=0.420 ap=0.007  test_pos=12
  [SEIZ] chb18_31  thr=-0.383 k=2  C=10 g=0.0001 p=10  roc=0.964 ap=0.765  test_pos=28
  [SEIZ] chb18_32  thr=-1.132 k=6  C=10 g=0.0001 p=10  roc=0.997 ap=0.907  test_pos=22
  [SEIZ] chb18_35  thr=0.279 k=1  C=10 g=0.0001 p=5  roc=0.817 ap=0.503  test_pos=27
  [SEIZ] chb18_36  thr=0.166 k=1  C=10 g=0.0001 p=10  roc=0.963 ap=0.653  test_pos=18
  [NON] 30 records scored (true LONO)
    chb18_29 done  (SEIZ)
    chb18_30 done  (SEIZ)
    chb18_31 done  (SEIZ)
    chb18_32 done  (SEIZ)
    chb18_35 done  (SEIZ)
    chb18_36 done  (SEIZ)
    chb18_01 done  (NON)
    chb18_02 done  (NON)
    chb18_03 done  (NON)
    chb18_04 done  (NON)
    chb18_05 done  (NON)
    chb18_06 done  (NON)
    chb18_07 done  (NON)
    chb18_08 done 

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.0001,10,chb18,45.634722,3,8,3,0.5,0.272727,0.352941,4.207323,0.175305,14.5,3,0.842008,0.584012


## chb16

In [ ]:
PATIENT = "chb16"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb19
[chb19] seizure=3 non-seizure=27
  [SEIZ] chb19_28  thr=-1.057 k=6  C=10 g=0.01 p=10  roc=0.696 ap=0.065  test_pos=31
  [SEIZ] chb19_29  thr=-0.999 k=4  C=1 g=0.0001 p=10  roc=0.946 ap=0.366  test_pos=31
  [SEIZ] chb19_30  thr=-1.000 k=8  C=1 g=0.0001 p=5  roc=0.718 ap=0.060  test_pos=33
  [NON] 27 records scored (true LONO)
    chb19_28 done  (SEIZ)
    chb19_29 done  (SEIZ)
    chb19_30 done  (SEIZ)
    chb19_01 done  (NON)
    chb19_02 done  (NON)
    chb19_03 done  (NON)
    chb19_04 done  (NON)
    chb19_05 done  (NON)
    chb19_06 done  (NON)
    chb19_07 done  (NON)
    chb19_08 done  (NON)
    chb19_09 done  (NON)
    chb19_10 done  (NON)
    chb19_11 done  (NON)
    chb19_12 done  (NON)
    chb19_13 done  (NON)
    chb19_14 done  (NON)
    chb19_15 done  (NON)
    chb19_16 done  (NON)
    chb19_17 done  (NON)
    chb19_18 done  (NON)
    chb19_19 done  (NON)
    chb19_20 done  (NON)
    chb19_21 done  (NON)
    chb

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.01,10,chb19,38.928472,0,9,3,0.0,0.0,0.0,5.548638,0.231193,NaN,0,0.786606,0.163649


In [ ]:
PATIENT = "chb17"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb20
[chb20] seizure=6 non-seizure=23
  [SEIZ] chb20_12  thr=0.143 k=1  C=10 g=0.001 p=10  roc=0.976 ap=0.336  test_pos=12
  [SEIZ] chb20_13  thr=-0.325 k=2  C=10 g=0.001 p=10  roc=0.991 ap=0.758  test_pos=28
  [SEIZ] chb20_14  thr=-1.203 k=6  C=10 g=0.0001 p=10  roc=0.952 ap=0.210  test_pos=15
  [SEIZ] chb20_15  thr=-1.749 k=8  C=10 g=0.01 p=5  roc=0.908 ap=0.539  test_pos=35
  [SEIZ] chb20_16  thr=-1.293 k=6  C=10 g=0.01 p=5  roc=0.987 ap=0.544  test_pos=14
  [SEIZ] chb20_68  thr=-1.419 k=6  C=10 g=0.001 p=5  roc=0.975 ap=0.304  test_pos=15
  [NON] 23 records scored (true LONO)
    chb20_12 done  (SEIZ)
    chb20_13 done  (SEIZ)
    chb20_14 done  (SEIZ)
    chb20_15 done  (SEIZ)
    chb20_16 done  (SEIZ)
    chb20_68 done  (SEIZ)
    chb20_01 done  (NON)
    chb20_02 done  (NON)
    chb20_03 done  (NON)
    chb20_04 done  (NON)
    chb20_05 done  (NON)
    chb20_06 done  (NON)
    chb20_07 done  (NON)
    chb20_08 done  (NON)

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb20,34.611111,8,5,0,1.0,0.615385,0.761905,3.467095,0.144462,26.75,8,0.964867,0.44862


In [ ]:
PATIENT = "chb18"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb21
[chb21] seizure=4 non-seizure=29
  [SEIZ] chb21_19  thr=-0.181 k=2  C=10 g=0.01 p=5  roc=0.980 ap=0.693  test_pos=22
  [SEIZ] chb21_20  thr=-0.244 k=2  C=10 g=0.001 p=5  roc=0.944 ap=0.649  test_pos=20
  [SEIZ] chb21_21  thr=-0.097 k=2  C=10 g=0.001 p=10  roc=0.962 ap=0.608  test_pos=32
  [SEIZ] chb21_22  thr=-0.023 k=1  C=10 g=0.01 p=10  roc=0.981 ap=0.484  test_pos=5
  [NON] 29 records scored (true LONO)
    chb21_19 done  (SEIZ)
    chb21_20 done  (SEIZ)
    chb21_21 done  (SEIZ)
    chb21_22 done  (SEIZ)
    chb21_01 done  (NON)
    chb21_02 done  (NON)
    chb21_03 done  (NON)
    chb21_04 done  (NON)
    chb21_05 done  (NON)
    chb21_06 done  (NON)
    chb21_07 done  (NON)
    chb21_08 done  (NON)
    chb21_09 done  (NON)
    chb21_10 done  (NON)
    chb21_11 done  (NON)
    chb21_12 done  (NON)
    chb21_13 done  (NON)
    chb21_14 done  (NON)
    chb21_15 done  (NON)
    chb21_16 done  (NON)
    chb21_17 done  (NON

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.01,10,chb21,42.496528,4,7,0,1.0,0.363636,0.533333,3.953264,0.164719,18.75,4,0.966606,0.608203


In [ ]:
PATIENT = "chb19"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb22
  [DROP complex] chb22_11_features.mat: 1 window(s) [np.int64(1206)]
[chb22] seizure=3 non-seizure=28
  [SEIZ] chb22_20  thr=0.384 k=1  C=10 g=0.001 p=5  roc=0.990 ap=0.829  test_pos=24
  [SEIZ] chb22_25  thr=-0.361 k=1  C=1 g=0.001 p=10  roc=0.918 ap=0.727  test_pos=30
  [SEIZ] chb22_38  thr=-0.367 k=1  C=1 g=0.01 p=5  roc=0.997 ap=0.943  test_pos=29
  [NON] 28 records scored (true LONO)
    chb22_20 done  (SEIZ)
    chb22_25 done  (SEIZ)
    chb22_38 done  (SEIZ)
    chb22_01 done  (NON)
    chb22_02 done  (NON)
    chb22_03 done  (NON)
    chb22_04 done  (NON)
    chb22_05 done  (NON)
    chb22_06 done  (NON)
    chb22_07 done  (NON)
    chb22_08 done  (NON)
    chb22_09 done  (NON)
    chb22_10 done  (NON)
    chb22_11 done  (NON)
    chb22_15 done  (NON)
    chb22_16 done  (NON)
    chb22_17 done  (NON)
    chb22_18 done  (NON)
    chb22_19 done  (NON)
    chb22_21 done  (NON)
    chb22_22 done  (NON)
    chb22_23 done

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb22,40.336111,3,1,0,1.0,0.75,0.857143,0.595,0.024792,29.5,3,0.968318,0.833064


## chb 20

In [ ]:
PATIENT = "chb20"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb23
[chb23] seizure=3 non-seizure=6
  [SEIZ] chb23_06  thr=-1.067 k=6  C=10 g=0.001 p=10  roc=0.977 ap=0.763  test_pos=46
  [SEIZ] chb23_08  thr=0.474 k=1  C=10 g=0.001 p=5  roc=0.980 ap=0.569  test_pos=28
  [SEIZ] chb23_09  thr=0.162 k=1  C=10 g=0.001 p=5  roc=0.964 ap=0.568  test_pos=99
  [NON] 6 records scored (true LONO)
    chb23_06 done  (SEIZ)
    chb23_08 done  (SEIZ)
    chb23_09 done  (SEIZ)
    chb23_07 done  (NON)
    chb23_10 done  (NON)
    chb23_16 done  (NON)
    chb23_17 done  (NON)
    chb23_19 done  (NON)
    chb23_20 done  (NON)
  EVENT: TP=7 FP=4 FN=0  sens=1.000 prec=0.636 f1=0.778  FP/day=2.961  lat=22.5s
  >>> SAVED [chb23] -> RESULTS_SUMMARY_chb23.csv

Wrote RESULTS_SUMMARY_chb23.csv

Pooled event-level:
 TP  FP  FN  sensitivity  precision   FP/day
  7   4   0          1.0   0.636364 2.960869


,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb23,32.422917,7,4,0,1.0,0.636364,0.777778,2.960869,0.12337,22.5,7,0.97388,0.63328


In [ ]:
PATIENT = "chb21"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


C / gamma / percentile auto-tuned per fold  |  patient = chb24
[chb24] seizure=12 non-seizure=10
  [SEIZ] chb24_01  thr=1.140 k=2  C=10 g=0.0001 p=10  roc=0.993 ap=0.607  test_pos=11
  [SEIZ] chb24_03  thr=0.709 k=2  C=10 g=0.001 p=10  roc=0.990 ap=0.831  test_pos=12
  [SEIZ] chb24_04  thr=0.917 k=4  C=10 g=0.0001 p=10  roc=0.944 ap=0.194  test_pos=13
  [SEIZ] chb24_06  thr=0.714 k=2  C=10 g=0.001 p=10  roc=0.943 ap=0.777  test_pos=10
  [SEIZ] chb24_07  thr=0.735 k=3  C=10 g=0.001 p=10  roc=0.910 ap=0.428  test_pos=9
  [SEIZ] chb24_09  thr=0.458 k=4  C=1 g=0.001 p=10  roc=1.000 ap=0.986  test_pos=8
  [SEIZ] chb24_11  thr=0.533 k=2  C=10 g=0.001 p=10  roc=0.550 ap=0.245  test_pos=28
  [SEIZ] chb24_13  thr=1.001 k=3  C=10 g=0.0001 p=10  roc=0.996 ap=0.609  test_pos=6
  [SEIZ] chb24_14  thr=0.775 k=2  C=10 g=0.001 p=10  roc=0.854 ap=0.307  test_pos=11
  [SEIZ] chb24_15  thr=0.861 k=2  C=10 g=0.001 p=10  roc=0.995 ap=0.853  test_pos=7
  [SEIZ] chb24_17  thr=0.835 k=2  C=10 g=0.001 p=10  ro

,C,gamma,percentile,patient,total_length_hours,TP,FP,FN,sensitivity,precision,f1,FP/day,FP/hour,latency_median,n_detected,mean_fold_roc_auc,mean_fold_ap
0,10,0.001,10,chb24,24.395139,2,3,10,0.166667,0.4,0.235294,2.951408,0.122975,22.5,2,0.930815,0.63662


In [ ]:
PATIENT = "chb22"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb23"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df


In [ ]:
PATIENT = "chb24"          # <-- the single patient to run

print(f"C / gamma / percentile auto-tuned per fold  |  patient = {PATIENT}")

SUMMARY_CSV = f"RESULTS_SUMMARY_{PATIENT}.csv"
SEGMENT_CSV = f"SEGMENT_PER_RECORD_{PATIENT}.csv"
EVENTS_CSV  = f"EVENTS_PER_RECORD_{PATIENT}.csv"
FP_CSV      = f"FALSE_POSITIVES_PER_RECORD_{PATIENT}.csv"

all_summaries, all_segments, all_events, all_fps = [], [], [], []

for patient in [PATIENT]:
    try:
        srow, segrows, evrows, fprows = run_patient(patient, cfg)

        if srow is not None:
            all_summaries.append(srow)
            all_segments.extend(segrows)
            all_events.extend(evrows)
            all_fps.extend(fprows)

        pd.DataFrame(all_summaries).to_csv(SUMMARY_CSV, index=False)
        pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
        pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
        pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

        print(f"  >>> SAVED [{patient}] -> {SUMMARY_CSV}", flush=True)

    except Exception as e:
        print(f"[{patient}] FAILED: {type(e).__name__}: {e}")
        continue

# final write
summary_df = pd.DataFrame(all_summaries)
summary_df.to_csv(SUMMARY_CSV, index=False)
pd.DataFrame(all_segments).to_csv(SEGMENT_CSV, index=False)
pd.DataFrame(all_events).to_csv(EVENTS_CSV, index=False)
pd.DataFrame(all_fps).to_csv(FP_CSV, index=False)

print(f"\nWrote {SUMMARY_CSV}")

# pooled metrics
if len(summary_df):
    TP = summary_df.TP.sum()
    FP = summary_df.FP.sum()
    FN = summary_df.FN.sum()
    total_hours = summary_df.total_length_hours.sum()

    pooled = pd.DataFrame([{
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "sensitivity": TP/(TP+FN) if (TP+FN) else float("nan"),
        "precision": TP/(TP+FP) if (TP+FP) else float("nan"),
        "FP/day": FP/(total_hours/24) if total_hours else float("nan"),
    }])

    print("\nPooled event-level:")
    print(pooled.to_string(index=False))

summary_df
